# My first FFN 

# Classification 
---
Types of classification: https://www.geeksforgeeks.org/machine-learning/getting-started-with-classification/

---

- Binary Classification

- Loading our binary classification data

In [1]:
import pandas as pd

In [2]:
data = pd.read_csv("Datasets/heart.csv")

data.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [3]:
# Seperating features and target as X and Y
X = data.drop("HeartDisease", axis=1)
y = data["HeartDisease"]

X.shape, y.shape



((918, 11), (918,))

In [4]:
X

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up
...,...,...,...,...,...,...,...,...,...,...,...
913,45,M,TA,110,264,0,Normal,132,N,1.2,Flat
914,68,M,ASY,144,193,1,Normal,141,N,3.4,Flat
915,57,M,ASY,130,131,0,Normal,115,Y,1.2,Flat
916,57,F,ATA,130,236,0,LVH,174,N,0.0,Flat


In [5]:
y

0      0
1      1
2      0
3      1
4      0
      ..
913    1
914    1
915    1
916    1
917    0
Name: HeartDisease, Length: 918, dtype: int64

In [6]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
col = ["Sex", "ChestPainType", "RestingECG", "ExerciseAngina", "ST_Slope"]

X[col] = X[col].apply(le.fit_transform)

X.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope
0,40,1,1,140,289,0,1,172,0,0.0,2
1,49,0,2,160,180,0,1,156,0,1.0,1
2,37,1,1,130,283,0,2,98,0,0.0,2
3,48,0,0,138,214,0,1,108,1,1.5,1
4,54,1,2,150,195,0,1,122,0,0.0,2


- Train test split data (80% training, 20% testing)

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((734, 11), (184, 11), (734,), (184,))

In [8]:
# Convert data to tensors

import torch

X_train_tensor = torch.tensor(X_train.values, dtype=torch.float)
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long)

- Building classification model 

In [9]:
import torch
import torch.nn as nn

In [10]:
model = nn.Sequential(
    # Layer 1
    nn.Linear(11, 22),
    nn.ReLU(),

    # Layer 2
    nn.Linear(22, 44),
    nn.ReLU(),

    # Layer 3
    nn.Linear(44, 22),
    nn.ReLU(),

    # Layer 4
    nn.Linear(22, 1),
    nn.Sigmoid()
)

print(model)

Sequential(
  (0): Linear(in_features=11, out_features=22, bias=True)
  (1): ReLU()
  (2): Linear(in_features=22, out_features=44, bias=True)
  (3): ReLU()
  (4): Linear(in_features=44, out_features=22, bias=True)
  (5): ReLU()
  (6): Linear(in_features=22, out_features=1, bias=True)
  (7): Sigmoid()
)


In [11]:
single_layer_example = nn.Linear(11, 22)

In [12]:
single_layer_example(X_train_tensor).shape

torch.Size([734, 22])

In [13]:
# Now decide out loss function for binary classification task

loss_function = nn.BCELoss()

# decide the optimizer we will be using
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [14]:
from sklearn.metrics import accuracy_score

In [15]:
# training loop for our model

for epoch in range(101):
    model.train() # Good practice
    out = model(X_train_tensor)
    loss = loss_function(out, y_train_tensor.unsqueeze(dim=1).float())

    optimizer.zero_grad() # Clears old gradients, NOT weights
    loss.backward()
    optimizer.step()

    # Evaluation Mode
    model.eval() 
    with torch.no_grad(): # Saves memory/time during testing
        y_pred = model(X_test_tensor)
        y_pred_class = (y_pred > 0.5).int()

    # Now calculate accuracy
    accuracy = accuracy_score(y_test_tensor.unsqueeze(dim=1).float().detach().numpy(), y_pred_class.detach().numpy())

    if epoch % 10 == 0:
        print("Current epoch: ", epoch, "| And current loss: ", loss.item(), "| Model Accuracy: ", accuracy)

Current epoch:  0 | And current loss:  0.7195420265197754 | Model Accuracy:  0.5815217391304348
Current epoch:  10 | And current loss:  0.6177991628646851 | Model Accuracy:  0.6195652173913043
Current epoch:  20 | And current loss:  0.5619205832481384 | Model Accuracy:  0.7119565217391305
Current epoch:  30 | And current loss:  0.5370540618896484 | Model Accuracy:  0.6684782608695652
Current epoch:  40 | And current loss:  0.524408221244812 | Model Accuracy:  0.7065217391304348
Current epoch:  50 | And current loss:  0.5055443048477173 | Model Accuracy:  0.7282608695652174
Current epoch:  60 | And current loss:  0.4812679886817932 | Model Accuracy:  0.717391304347826
Current epoch:  70 | And current loss:  0.44865795969963074 | Model Accuracy:  0.7717391304347826
Current epoch:  80 | And current loss:  0.41956669092178345 | Model Accuracy:  0.7717391304347826
Current epoch:  90 | And current loss:  0.3964127004146576 | Model Accuracy:  0.7989130434782609
Current epoch:  100 | And curre

# Classic Pytorch Model(Recommended) # IMP

In [16]:
# Example class creation - demo practice
class ExampleClass:
    def __init__(self):
        print("Hello")
        # Use of self key on variable
        self.a = 1
        print(self.a)
        
    # Method
    def forward(self, name):
        print("Your name is:", name)
        # Update variable having self key
        self.a = 100
        print(self.a)
        

# object = class()
a = ExampleClass()

# calling a method for class ExampleClass with name as input parameter
a.forward(name="Aish")


Hello
1
Your name is: Aish
100


- To create a Deep learning AI model in pytorch we use classes.

In [17]:
class BCModel(nn.Module):
    def __init__(self):
        super(BCModel, self).__init__() # Calling nn.Module constructor from BCModel constructor

        # Model layer 
        self.fc1 = nn.Linear(11, 22)
        self.fc2 = nn.Linear(22, 44)
        self.fc3 = nn.Linear(44, 22)
        self.fc4 = nn.Linear(22, 1)

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out = self.fc1(x)
        out = self.fc2(out)
        out = self.fc3(out)
        out = self.fc4(out)
        out = self.sigmoid(out)

        return out
        

In [18]:
# object = class name
my_model = BCModel()
my_model.to(device=torch.device("cuda"))

BCModel(
  (fc1): Linear(in_features=11, out_features=22, bias=True)
  (fc2): Linear(in_features=22, out_features=44, bias=True)
  (fc3): Linear(in_features=44, out_features=22, bias=True)
  (fc4): Linear(in_features=22, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

In [19]:
# loss function for Binary Classification - Binary Cross Entropy
loss_function = nn.BCELoss()


In [20]:
# Optimizer - Adam
optimizer = torch.optim.Adam(my_model.parameters(), lr=0.001)

In [21]:
# training Binary Classification Model
X_train_tensor = X_train_tensor.to(device=torch.device("cuda"))
y_train_tensor = y_train_tensor.to(device=torch.device("cuda"))
epochs = 100

for epoch in range(epochs):
    # put model in training mode 
    my_model.train()

    # y_train target for X_train
    y_target = y_train_tensor.unsqueeze(dim=1).float()

    # predict value
    out = my_model(X_train_tensor)
    # calculate loss between predicted value and actual value of input x
    loss = loss_function(out, y_target)

    # now adjust the values of weights and biases in neurons using optimizer and loss
    optimizer.zero_grad()
    loss.backward() # do back-propogation
    optimizer.step() # update the weights

    if epoch % 10 == 0:
        print("Epochs: ", epoch, "| Loss: ", loss.item())
        
    

Epochs:  0 | Loss:  1.9880547523498535
Epochs:  10 | Loss:  0.5976303219795227
Epochs:  20 | Loss:  0.6117278933525085
Epochs:  30 | Loss:  0.5987666249275208
Epochs:  40 | Loss:  0.5770595073699951
Epochs:  50 | Loss:  0.5654287934303284
Epochs:  60 | Loss:  0.5588198304176331
Epochs:  70 | Loss:  0.5541031956672668
Epochs:  80 | Loss:  0.5504275560379028
Epochs:  90 | Loss:  0.5468243360519409


## Applying Activation function

In [22]:
class BCModelAF(nn.Module):
    def __init__(self):
        super(BCModelAF, self).__init__()
    
        # layer 
        self.fc1 = nn.Linear(11, 22)
        self.relu1 = nn.ReLU()
    
        self.fc2 = nn.Linear(22, 44)
        self.relu2 = nn.ReLU()
        
        self.fc3 = nn.Linear(44, 22)
        self.relu3 = nn.ReLU()
        
        self.fc4 = nn.Linear(22, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
     # layer 1 of FNN 
      out = self.fc1(x)
      out = self.relu1(out)

    # layer 2 of FNN 
      out = self.fc2(out)
      out = self.relu2(out)

    # layer 3 of FNN
      out = self.fc3(out)
      out = self.relu3(out)

    # layer 4 of FNN
      out = self.fc4(out)
      out = self.sigmoid(out)
        
      return out

In [23]:
my_model = BCModelAF()

In [24]:
my_model.to(device=torch.device("cuda"))

BCModelAF(
  (fc1): Linear(in_features=11, out_features=22, bias=True)
  (relu1): ReLU()
  (fc2): Linear(in_features=22, out_features=44, bias=True)
  (relu2): ReLU()
  (fc3): Linear(in_features=44, out_features=22, bias=True)
  (relu3): ReLU()
  (fc4): Linear(in_features=22, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

In [25]:
# loss function
loss_function = nn.BCELoss()

# optimizer
optimizer = torch.optim.Adam(my_model.parameters(), lr=0.001)


In [26]:
X_train_tensor = X_train_tensor.to(device=torch.device("cuda"))
y_train_tensor = y_train_tensor.to(device=torch.device("cuda"))

epoches = 101

for epoch in range(epoches):
    # put model on training mode 
    model.train()

    y_target = y_train_tensor.unsqueeze(dim=1).float()

    out = my_model(X_train_tensor)
    loss = loss_function(out, y_target)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print("Epochs: ", epoch, "| Loss:", loss.item())

Epochs:  0 | Loss: 2.50042462348938
Epochs:  10 | Loss: 0.8105324506759644
Epochs:  20 | Loss: 0.6266024112701416
Epochs:  30 | Loss: 0.5655815005302429
Epochs:  40 | Loss: 0.5526850819587708
Epochs:  50 | Loss: 0.5463274121284485
Epochs:  60 | Loss: 0.5398521423339844
Epochs:  70 | Loss: 0.5358458161354065
Epochs:  80 | Loss: 0.533184826374054
Epochs:  90 | Loss: 0.5307677388191223
Epochs:  100 | Loss: 0.528689980506897


# Monitoring model training using wandb(Weights and Biases)

In [27]:
import random

import wandb

# Start a new wandb run to track this script.
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="aishwaryapatil845-aish-com",
    # Set the wandb project where this run will be logged.
    project="my-awesome-project",
    # Track hyperparameters and run metadata.
    config={
        "learning_rate": 0.001,
        "architecture": "FNN",
        "dataset": "Heart_Disease Dataset",
        "epochs": 500,
    },
)




wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/aishwarya/.netrc.
wandb: Currently logged in as: aishwaryapatil845 (aishwaryapatil845-aish-com) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [28]:
X_train_tensor = X_train_tensor.to(device=torch.device("cuda"))
y_train_tensor = y_train_tensor.to(device=torch.device("cuda"))

epoches = 500

for epoch in range(epoches):
    # put model on training mode 
    model.train()

    y_target = y_train_tensor.unsqueeze(dim=1).float()

    out = my_model(X_train_tensor)
    loss = loss_function(out, y_target)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print("Epochs: ", epoch, "| Loss:", loss.item())
        run.log({"loss": loss.item(), "epoch": epoch})

Epochs:  0 | Loss: 0.5284876823425293
Epochs:  10 | Loss: 0.5265548825263977
Epochs:  20 | Loss: 0.5245828628540039
Epochs:  30 | Loss: 0.5223883986473083
Epochs:  40 | Loss: 0.5201917290687561
Epochs:  50 | Loss: 0.5179036855697632
Epochs:  60 | Loss: 0.5155324935913086
Epochs:  70 | Loss: 0.5128015279769897
Epochs:  80 | Loss: 0.5094513893127441
Epochs:  90 | Loss: 0.5057796835899353
Epochs:  100 | Loss: 0.5008073449134827
Epochs:  110 | Loss: 0.49398699402809143
Epochs:  120 | Loss: 0.4861326217651367
Epochs:  130 | Loss: 0.476630836725235
Epochs:  140 | Loss: 0.4641619622707367
Epochs:  150 | Loss: 0.4525696337223053
Epochs:  160 | Loss: 0.43972352147102356
Epochs:  170 | Loss: 0.42684727907180786
Epochs:  180 | Loss: 0.4114748239517212
Epochs:  190 | Loss: 0.400212824344635
Epochs:  200 | Loss: 0.391832172870636
Epochs:  210 | Loss: 0.3825966715812683
Epochs:  220 | Loss: 0.37439095973968506
Epochs:  230 | Loss: 0.3685363531112671
Epochs:  240 | Loss: 0.36439675092697144
Epochs:  

## Comparing models using wandb

In [29]:
# Start a new wandb run to track this script.
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="aishwaryapatil845-aish-com",
    # Set the wandb project where this run will be logged.
    project="HeartModel",
    name="Epoch100",
    # Track hyperparameters and run metadata.
    config={
        "learning_rate": 0.001,
        "architecture": "FNN",
        "dataset": "Heart_Disease Dataset",
        "epochs": 100,
    },
)


X_train_tensor = X_train_tensor.to(device=torch.device("cuda"))
y_train_tensor = y_train_tensor.to(device=torch.device("cuda"))

epoches = 100

for epoch in range(epoches):
    # put model on training mode 
    model.train()

    y_target = y_train_tensor.unsqueeze(dim=1).float()

    out = my_model(X_train_tensor)
    loss = loss_function(out, y_target)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print("Epochs: ", epoch, "| Loss:", loss.item())
        run.log({"loss": loss.item(), "epoch": epoch})

epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
loss,██████▇▇▇▇▇▆▆▅▅▄▄▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁
epoch,490
loss,0.32006


Epochs:  0 | Loss: 0.3184574544429779
Epochs:  10 | Loss: 0.3204098343849182
Epochs:  20 | Loss: 0.3166709542274475
Epochs:  30 | Loss: 0.31319665908813477
Epochs:  40 | Loss: 0.3121483027935028
Epochs:  50 | Loss: 0.30987465381622314
Epochs:  60 | Loss: 0.3110084533691406
Epochs:  70 | Loss: 0.30916672945022583
Epochs:  80 | Loss: 0.3074396848678589
Epochs:  90 | Loss: 0.30538448691368103


In [30]:
# 500 epoch

# Start a new wandb run to track this script.
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="aishwaryapatil845-aish-com",
    # Set the wandb project where this run will be logged.
    project="HeartModel",
    name="Epoch500",
    # Track hyperparameters and run metadata.
    config={
        "learning_rate": 0.001,
        "architecture": "FNN",
        "dataset": "Heart_Disease Dataset",
        "epochs": 500,
    },
)


X_train_tensor = X_train_tensor.to(device=torch.device("cuda"))
y_train_tensor = y_train_tensor.to(device=torch.device("cuda"))

epoches = 500

for epoch in range(epoches):
    # put model on training mode 
    model.train()

    y_target = y_train_tensor.unsqueeze(dim=1).float()

    out = my_model(X_train_tensor)
    loss = loss_function(out, y_target)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print("Epochs: ", epoch, "| Loss:", loss.item())
        run.log({"loss": loss.item(), "epoch": epoch})

epoch,▁▂▃▃▄▅▆▆▇█
loss,▇█▆▅▄▃▄▃▂▁
epoch,90
loss,0.30538


Epochs:  0 | Loss: 0.3033774793148041
Epochs:  10 | Loss: 0.302335262298584
Epochs:  20 | Loss: 0.30162763595581055
Epochs:  30 | Loss: 0.3009790778160095
Epochs:  40 | Loss: 0.2991017997264862
Epochs:  50 | Loss: 0.2988583445549011
Epochs:  60 | Loss: 0.2998206615447998
Epochs:  70 | Loss: 0.2968383729457855
Epochs:  80 | Loss: 0.29783186316490173
Epochs:  90 | Loss: 0.2957739233970642
Epochs:  100 | Loss: 0.2932076156139374
Epochs:  110 | Loss: 0.2918928563594818
Epochs:  120 | Loss: 0.3009902536869049
Epochs:  130 | Loss: 0.28955313563346863
Epochs:  140 | Loss: 0.2886711061000824
Epochs:  150 | Loss: 0.2869030237197876
Epochs:  160 | Loss: 0.286542683839798
Epochs:  170 | Loss: 0.28454524278640747
Epochs:  180 | Loss: 0.2841556966304779
Epochs:  190 | Loss: 0.28456225991249084
Epochs:  200 | Loss: 0.28252914547920227
Epochs:  210 | Loss: 0.28191351890563965
Epochs:  220 | Loss: 0.2822347581386566
Epochs:  230 | Loss: 0.281317800283432
Epochs:  240 | Loss: 0.27852076292037964
Epochs

In [31]:
# 500 epoch

# Start a new wandb run to track this script.
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="aishwaryapatil845-aish-com",
    # Set the wandb project where this run will be logged.
    project="HeartModel",
    name="Epoch500",
    # Track hyperparameters and run metadata.
    config={
        "learning_rate": 0.001,
        "architecture": "FNN",
        "dataset": "Heart_Disease Dataset",
        "epochs": 500,
    },
)


X_train_tensor = X_train_tensor.to(device=torch.device("cuda"))
y_train_tensor = y_train_tensor.to(device=torch.device("cuda"))

epoches = 500

for epoch in range(epoches):
    # put model on training mode 
    model.train()

    y_target = y_train_tensor.unsqueeze(dim=1).float()

    out = my_model(X_train_tensor)
    loss = loss_function(out, y_target)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print("Epochs: ", epoch, "| Loss:", loss.item())
        run.log({"loss": loss.item(), "epoch": epoch})

epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
loss,████▇▇▇▇▇▆▆▆▅▅▅▄▄▄▄▄▄▄▄▃▄▃▃▃▃▃▃▂▂▂▂▂▂▁▁▁
epoch,490
loss,0.2617


Epochs:  0 | Loss: 0.26198047399520874
Epochs:  10 | Loss: 0.260463684797287
Epochs:  20 | Loss: 0.2597907483577728
Epochs:  30 | Loss: 0.259337842464447
Epochs:  40 | Loss: 0.2616049647331238
Epochs:  50 | Loss: 0.26204001903533936
Epochs:  60 | Loss: 0.257567822933197
Epochs:  70 | Loss: 0.25822341442108154
Epochs:  80 | Loss: 0.25633373856544495
Epochs:  90 | Loss: 0.2564922869205475
Epochs:  100 | Loss: 0.25811317563056946
Epochs:  110 | Loss: 0.2580764591693878
Epochs:  120 | Loss: 0.2555586099624634
Epochs:  130 | Loss: 0.25606855750083923
Epochs:  140 | Loss: 0.2561042904853821
Epochs:  150 | Loss: 0.25242581963539124
Epochs:  160 | Loss: 0.25202345848083496
Epochs:  170 | Loss: 0.2536800801753998
Epochs:  180 | Loss: 0.2546515166759491
Epochs:  190 | Loss: 0.25368088483810425
Epochs:  200 | Loss: 0.2503965497016907
Epochs:  210 | Loss: 0.2498530149459839
Epochs:  220 | Loss: 0.2508666217327118
Epochs:  230 | Loss: 0.24935393035411835
Epochs:  240 | Loss: 0.2494853138923645
Epoc

In [32]:
# Start a new wandb run to track this script.
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="aishwaryapatil845-aish-com",
    # Set the wandb project where this run will be logged.
    project="HeartModel",
    name="Epoch1000",
    # Track hyperparameters and run metadata.
    config={
        "learning_rate": 0.001,
        "architecture": "FNN",
        "dataset": "Heart_Disease Dataset",
        "epochs": 1000,
    },
)


X_train_tensor = X_train_tensor.to(device=torch.device("cuda"))
y_train_tensor = y_train_tensor.to(device=torch.device("cuda"))

epoches = 1000

for epoch in range(epoches):
    # put model on training mode 
    model.train()

    y_target = y_train_tensor.unsqueeze(dim=1).float()

    out = my_model(X_train_tensor)
    loss = loss_function(out, y_target)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print("Epochs: ", epoch, "| Loss:", loss.item())
        run.log({"loss": loss.item(), "epoch": epoch})

epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
loss,██▇▇█▇▇▆▆▇▆▆▆▅▅▆▆▅▄▅▄▄▄▇▅▄▃▃▃▃▂▃▃▃▂▂▁▁▁▁
epoch,490
loss,0.23773


Epochs:  0 | Loss: 0.24170994758605957
Epochs:  10 | Loss: 0.2402704656124115
Epochs:  20 | Loss: 0.23681452870368958
Epochs:  30 | Loss: 0.235365092754364
Epochs:  40 | Loss: 0.23556821048259735
Epochs:  50 | Loss: 0.23726406693458557
Epochs:  60 | Loss: 0.23641343414783478
Epochs:  70 | Loss: 0.23499274253845215
Epochs:  80 | Loss: 0.24029208719730377
Epochs:  90 | Loss: 0.2336844801902771
Epochs:  100 | Loss: 0.233940988779068
Epochs:  110 | Loss: 0.23317977786064148
Epochs:  120 | Loss: 0.23275858163833618
Epochs:  130 | Loss: 0.2318905144929886
Epochs:  140 | Loss: 0.23457539081573486
Epochs:  150 | Loss: 0.23823633790016174
Epochs:  160 | Loss: 0.23268365859985352
Epochs:  170 | Loss: 0.23271331191062927
Epochs:  180 | Loss: 0.23289954662322998
Epochs:  190 | Loss: 0.23037037253379822
Epochs:  200 | Loss: 0.23216459155082703
Epochs:  210 | Loss: 0.22867931425571442
Epochs:  220 | Loss: 0.2296086847782135
Epochs:  230 | Loss: 0.2294306457042694
Epochs:  240 | Loss: 0.2282876074314

### Multiclass Classification
- Has more than two classes 
- Using iris dataset which has more than 2 classes (3 class)

In [33]:
# Load dataset using scikit-learn
from sklearn.datasets import load_iris

data = load_iris()
data.keys()


dict_keys(['data', 'target', 'frame', 'target_names', 'DESCR', 'feature_names', 'filename', 'data_module'])

- Lets check number of classes (target names)

In [34]:
print("Target names(classes): ", data["target_names"], "\nTotal number of classes: ", len(data["target_names"]))

Target names(classes):  ['setosa' 'versicolor' 'virginica'] 
Total number of classes:  3


In [35]:
X = data["data"] # input
y = data["target"] # output

In [36]:
X.shape, y.shape

((150, 4), (150,))

In [37]:
# data type of X and y
type(X), type(y)

(numpy.ndarray, numpy.ndarray)

- So there are numpy array therefore they can't work with our pytorch model
- Lets connect these X and y into tensors

In [38]:
import torch

In [39]:
device = "cuda" if torch.cuda.is_available() else "cpu"

X = torch.tensor(X, dtype=torch.float32).to(device)
y = torch.tensor(y, dtype=torch.long).to(device)

In [40]:
type(X), type(y)

(torch.Tensor, torch.Tensor)

- lets unsqueeze `y`

In [41]:
y.size()

torch.Size([150])

In [42]:
#y = y.unsqueeze(dim=1).to(device=torch.device("cuda"))
#y.size()

- Here we have converted our numpy array dataset into tensor
- Now lets split our data into 80 and 20 percent ratio using scikit-learn `train_test_split`

In [43]:
from sklearn.model_selection import train_test_split

In [44]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [45]:
X_train.size(), X_test.size(), y_train.size(), y_test.size()

(torch.Size([120, 4]),
 torch.Size([30, 4]),
 torch.Size([120]),
 torch.Size([30]))

In [46]:
X_train.shape

torch.Size([120, 4])

In [47]:
# Model 
class MCCModel(nn.Module):
    def __init__(self):
        super(MCCModel, self).__init__()

        # layer 
        self.fc1 = nn.Linear(4, 8)
        self.fc2 = nn.Linear(8, 16)
        self.fc3 = nn.Linear(16, 8)
        self.fc4 = nn.Linear(8, 3)

    def forward(self, x):
        x = self.fc1(x)
        x = self.fc2(x)
        x = self.fc3(x)
        x = self.fc4(x)

        return x 

In [48]:
multiclass_model = MCCModel()
multiclass_model.to(device=torch.device("cuda"))

MCCModel(
  (fc1): Linear(in_features=4, out_features=8, bias=True)
  (fc2): Linear(in_features=8, out_features=16, bias=True)
  (fc3): Linear(in_features=16, out_features=8, bias=True)
  (fc4): Linear(in_features=8, out_features=3, bias=True)
)

In [49]:
# loss function 
loss_function = nn.CrossEntropyLoss()

# optimizer 
optimizer = torch.optim.Adam(multiclass_model.parameters(), lr=0.001)


In [50]:
# training loop 
epoches = 100 

for epoch in range(epoches):
    multiclass_model.train() # put model in training mode

    out = multiclass_model(X_train)
    loss = loss_function(out, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print("Epochs: ", epoch, "| Loss: ", loss.item())
    

Epochs:  0 | Loss:  1.1305102109909058
Epochs:  10 | Loss:  1.0407904386520386
Epochs:  20 | Loss:  0.9965863823890686
Epochs:  30 | Loss:  0.9453080296516418
Epochs:  40 | Loss:  0.8816829919815063
Epochs:  50 | Loss:  0.804806649684906
Epochs:  60 | Loss:  0.7177504897117615
Epochs:  70 | Loss:  0.6317576169967651
Epochs:  80 | Loss:  0.558877170085907
Epochs:  90 | Loss:  0.503184974193573


## Evaluate Model 

In [51]:
from sklearn.metrics import classification_report 

multiclass_model.eval()

# 1. Get the raw predictions (logists)
y_pred_values = multiclass_model(X_test)

# 2. Find the predicted class (argmax along dimension 1)
# 3. Move the result to the CPU, detach, and convert to numpy
y_pred = y_pred_values.argmax(dim=1).cpu().detach().numpy()

In [52]:
print(classification_report(y_test.to(device=torch.device("cpu")), y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      0.33      0.50         9
           2       0.65      1.00      0.79        11

    accuracy                           0.80        30
   macro avg       0.88      0.78      0.76        30
weighted avg       0.87      0.80      0.77        30



# Visualizing Multi-Class Classifcation using Wandb 

In [53]:
from sklearn.metrics import accuracy_score

In [54]:
accuracy_score(y_test.to(device=torch.device("cpu")), y_pred)

0.8

In [55]:
# Start a new wandb run to track this script.
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="aishwaryapatil845-aish-com",
    # Set the wandb project where this run will be logged.
    project="Iris_Classification",
    name="IrisEpoch1000",
    # Track hyperparameters and run metadata.
    config={
        "learning_rate": 0.001,
        "architecture": "FNN",
        "dataset": "Iris",
        "epochs": 1000,
    },
)

# training loop 
epoches = 100 

for epoch in range(epoches):
    multiclass_model.train() # put model in training mode

    out = multiclass_model(X_train)
    loss = loss_function(out, y_train)

    # accuracy 
    multiclass_model.eval()
    y_pred_values = multiclass_model(X_test)
    y_pred = y_pred_values.argmax(dim=1).cpu().numpy()
    accuracy = accuracy_score(y_test.to(device=torch.device("cpu")), y_pred)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print("Epochs: ", epoch, "| Loss: ", loss.item())
        run.log({"loss": loss.item(), "epoch": epoch, "accuracy": accuracy}) # send data to wandb

epoch,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇██
loss,█▇▇▇▇▇▆▆▆▆▅▅▆▅▄▅▅▄▆▄▃▅▃▃▄▄▃▂▂▂▂▂▂▂▂▂▂▂▁▁
epoch,990
loss,0.20741


Epochs:  0 | Loss:  0.46003276109695435
Epochs:  10 | Loss:  0.4234277606010437
Epochs:  20 | Loss:  0.38946911692619324
Epochs:  30 | Loss:  0.3561094403266907
Epochs:  40 | Loss:  0.32257524132728577
Epochs:  50 | Loss:  0.28902336955070496
Epochs:  60 | Loss:  0.25626426935195923
Epochs:  70 | Loss:  0.2254355102777481
Epochs:  80 | Loss:  0.19762693345546722
Epochs:  90 | Loss:  0.17356547713279724


In [56]:
# epoch 260
# --------------------------------Load model again - all values zero
multiclass_model = MCCModel()
multiclass_model.to(device=torch.device("cuda"))
# --------------------------------Load optimizer again - all values zero
optimizer = torch.optim.Adam(multiclass_model.parameters(), lr=0.001)


# Start a new wandb run to track this script.
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="aishwaryapatil845-aish-com",
    # Set the wandb project where this run will be logged.
    project="Iris_Classification",
    name="IrisEpoch260",
    # Track hyperparameters and run metadata.
    config={
        "learning_rate": 0.001,
        "architecture": "FNN",
        "dataset": "Iris",
        "epochs": 260,
    },
)

# training loop 
epoches = 260 

for epoch in range(epoches):
    multiclass_model.train() # put model in training mode

    out = multiclass_model(X_train)
    loss = loss_function(out, y_train)

    # accuracy 
    multiclass_model.eval()
    y_pred_values = multiclass_model(X_test)
    y_pred = y_pred_values.argmax(dim=1).cpu().numpy()
    accuracy = accuracy_score(y_test.to(device=torch.device("cpu")), y_pred)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print("Epochs: ", epoch, "| Loss: ", loss.item())
        run.log({"loss": loss.item(), "epoch": epoch, "accuracy": accuracy}) # send data to wandb

accuracy,▁▂▅▅▇███▇▇
epoch,▁▂▃▃▄▅▆▆▇█
loss,█▇▆▅▅▄▃▂▂▁
accuracy,0.93333
epoch,90
loss,0.17357


Epochs:  0 | Loss:  1.270850658416748
Epochs:  10 | Loss:  1.136263132095337
Epochs:  20 | Loss:  1.0761393308639526
Epochs:  30 | Loss:  1.0416831970214844
Epochs:  40 | Loss:  1.0040773153305054
Epochs:  50 | Loss:  0.9541037678718567
Epochs:  60 | Loss:  0.886798083782196
Epochs:  70 | Loss:  0.8011264801025391
Epochs:  80 | Loss:  0.7042074799537659
Epochs:  90 | Loss:  0.6114954352378845
Epochs:  100 | Loss:  0.5362554788589478
Epochs:  110 | Loss:  0.4796319305896759
Epochs:  120 | Loss:  0.43470922112464905
Epochs:  130 | Loss:  0.3949300944805145
Epochs:  140 | Loss:  0.3567330539226532
Epochs:  150 | Loss:  0.3189403712749481
Epochs:  160 | Loss:  0.2818986177444458
Epochs:  170 | Loss:  0.2467503547668457
Epochs:  180 | Loss:  0.21477708220481873
Epochs:  190 | Loss:  0.18690437078475952
Epochs:  200 | Loss:  0.16348224878311157
Epochs:  210 | Loss:  0.1443348377943039
Epochs:  220 | Loss:  0.12895403802394867
Epochs:  230 | Loss:  0.1167018935084343
Epochs:  240 | Loss:  0.1

In [57]:
# epoch 500
# --------------------------------Load model again - all values zero
multiclass_model = MCCModel()
multiclass_model.to(device=torch.device("cuda"))
# --------------------------------Load optimizer again - all values zero
optimizer = torch.optim.Adam(multiclass_model.parameters(), lr=0.001)


# Start a new wandb run to track this script.
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="aishwaryapatil845-aish-com",
    # Set the wandb project where this run will be logged.
    project="Iris_Classification",
    name="IrisEpoch500",
    # Track hyperparameters and run metadata.
    config={
        "learning_rate": 0.001,
        "architecture": "FNN",
        "dataset": "Iris",
        "epochs": 500,
    },
)

# training loop 
epoches = 500

for epoch in range(epoches):
    multiclass_model.train() # put model in training mode

    out = multiclass_model(X_train)
    loss = loss_function(out, y_train)

    # accuracy 
    multiclass_model.eval()
    y_pred_values = multiclass_model(X_test)
    y_pred = y_pred_values.argmax(dim=1).cpu().numpy()
    accuracy = accuracy_score(y_test.to(device=torch.device("cpu")), y_pred)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print("Epochs: ", epoch, "| Loss: ", loss.item())
        run.log({"loss": loss.item(), "epoch": epoch, "accuracy": accuracy}) # send data to wandb

accuracy,▁▁▂▅▅▅▅▅▆▆▆▇▇█████████████
epoch,▁▁▂▂▂▂▃▃▃▄▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
loss,█▇▇▇▆▆▆▅▅▄▄▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁
accuracy,0.96667
epoch,250
loss,0.09916


Epochs:  0 | Loss:  1.207092523574829
Epochs:  10 | Loss:  1.0975100994110107
Epochs:  20 | Loss:  1.044442892074585
Epochs:  30 | Loss:  1.0036017894744873
Epochs:  40 | Loss:  0.9508341550827026
Epochs:  50 | Loss:  0.88564133644104
Epochs:  60 | Loss:  0.8086678385734558
Epochs:  70 | Loss:  0.7243567109107971
Epochs:  80 | Loss:  0.6424597501754761
Epochs:  90 | Loss:  0.5722676515579224
Epochs:  100 | Loss:  0.5168723464012146
Epochs:  110 | Loss:  0.4730142056941986
Epochs:  120 | Loss:  0.43558165431022644
Epochs:  130 | Loss:  0.4007505774497986
Epochs:  140 | Loss:  0.3663013279438019
Epochs:  150 | Loss:  0.3313642740249634
Epochs:  160 | Loss:  0.296158105134964
Epochs:  170 | Loss:  0.2617080509662628
Epochs:  180 | Loss:  0.22944436967372894
Epochs:  190 | Loss:  0.20069068670272827
Epochs:  200 | Loss:  0.17624053359031677
Epochs:  210 | Loss:  0.15621954202651978
Epochs:  220 | Loss:  0.14023104310035706
Epochs:  230 | Loss:  0.12761814892292023
Epochs:  240 | Loss:  0.1

# Retraining model using ReLU activation functions 

In [58]:
class MCCModel(nn.Module):
    def __init__(self):
        super(MCCModel, self).__init__()

        # layer 
        self.fc1 = nn.Linear(4, 8)
        self.relu1 = nn.ReLU()

        self.fc2 = nn.Linear(8, 16)
        self.relu2 = nn.ReLU()

        self.fc3 = nn.Linear(16, 8)
        self.relu3 = nn.ReLU()

        self.fc4 = nn.Linear(8, 3)

    def forward(self, x):
        x = self.relu1(self.fc1(x)) # input layer

        # hidden layers
        x = self.relu2(self.fc2(x))
        x = self.relu3(self.fc3(x))

        # out put layer 
        x = self.fc4(x)
        return x

In [59]:
model = MCCModel()
model.to(device=torch.device("cuda"))
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# epoch 500
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="aishwaryapatil845-aish-com",
    # Set the wandb project where this run will be logged.
    project="Iris_Classification",
    name="IrisEpoch500 with ReLU",
    # Track hyperparameters and run metadata.
    config={
        "learning_rate": 0.001,
        "architecture": "FNN",
        "dataset": "Iris",
        "epochs": 500,
    },
)

# training loop 
epoches = 500

for epoch in range(epoches):
    model.train() # put model in training mode

    out = model(X_train)
    loss = loss_function(out, y_train)

    # accuracy 
    model.eval()
    y_pred_values = model(X_test)
    y_pred = y_pred_values.argmax(dim=1).cpu().numpy()
    accuracy = accuracy_score(y_test.to(device=torch.device("cpu")), y_pred)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print("Epochs: ", epoch, "| Loss: ", loss.item())
        run.log({"loss": loss.item(), "epoch": epoch, "accuracy": accuracy}) # send data to wandb


accuracy,▁▁▁▁▅▅▅▅▅▅▆▇▇▇██████████████████████████
epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
loss,█▇▇▇▆▅▅▄▄▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
accuracy,1
epoch,490
loss,0.06574


Epochs:  0 | Loss:  1.0953552722930908
Epochs:  10 | Loss:  1.0575357675552368
Epochs:  20 | Loss:  1.0202786922454834
Epochs:  30 | Loss:  0.9794765710830688
Epochs:  40 | Loss:  0.9301766753196716
Epochs:  50 | Loss:  0.870575487613678
Epochs:  60 | Loss:  0.803304135799408
Epochs:  70 | Loss:  0.7331991195678711
Epochs:  80 | Loss:  0.6655758023262024
Epochs:  90 | Loss:  0.6039450168609619
Epochs:  100 | Loss:  0.5499501824378967
Epochs:  110 | Loss:  0.500845193862915
Epochs:  120 | Loss:  0.4603062868118286
Epochs:  130 | Loss:  0.42454367876052856
Epochs:  140 | Loss:  0.39184829592704773
Epochs:  150 | Loss:  0.3609100580215454
Epochs:  160 | Loss:  0.33072566986083984
Epochs:  170 | Loss:  0.3008645474910736
Epochs:  180 | Loss:  0.27137279510498047
Epochs:  190 | Loss:  0.24272090196609497
Epochs:  200 | Loss:  0.21563301980495453
Epochs:  210 | Loss:  0.1908538043498993
Epochs:  220 | Loss:  0.1689155399799347
Epochs:  230 | Loss:  0.15011148154735565
Epochs:  240 | Loss:  0

# Applying Dropout

In [60]:
class MCCModel(nn.Module):
    def __init__(self):
        super(MCCModel, self).__init__()

        # layer 
        self.fc1 = nn.Linear(4, 8)
        self.relu1 = nn.ReLU()

        self.fc2 = nn.Linear(8, 16)
        self.relu2 = nn.ReLU()
        self.do1 = nn.Dropout(0.2)

        self.fc3 = nn.Linear(16, 8)
        self.relu3 = nn.ReLU()
        self.do2 = nn.Dropout(0.1)

        self.fc4 = nn.Linear(8, 3)

    def forward(self, x):
        x = self.relu1(self.fc1(x)) # input layer

        # hidden layers
        x = self.relu2(self.fc2(x))
        x = self.do1(x)
        
        x = self.relu3(self.fc3(x))
        x = self.do2(x)
        
        # out put layer 
        x = self.fc4(x)
        return x

In [61]:
model = MCCModel()
model.to(device=torch.device("cuda"))
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# epoch 500
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="aishwaryapatil845-aish-com",
    # Set the wandb project where this run will be logged.
    project="Iris_Classification",
    name="IrisEpoch500 with ReLU and Dropout",
    # Track hyperparameters and run metadata.
    config={
        "learning_rate": 0.001,
        "architecture": "FNN",
        "dataset": "Iris",
        "epochs": 500,
    },
)

# training loop 
epoches = 500

for epoch in range(epoches):
    model.train() # put model in training mode

    out = model(X_train)
    loss = loss_function(out, y_train)

    # accuracy 
    model.eval()
    y_pred_values = model(X_test)
    y_pred = y_pred_values.argmax(dim=1).cpu().detach().numpy()
    accuracy = accuracy_score(y_test.to(device=torch.device("cpu")), y_pred)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print("Epochs: ", epoch, "| Loss: ", loss.item())
        run.log({"loss": loss.item(), "epoch": epoch, "accuracy": accuracy}) # send data to wandb

accuracy,▁▄▆▆▅▅▅▅▅▆▇█████████████████████████████
epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
loss,██▇▇▇▆▆▅▅▄▄▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
accuracy,0.96667
epoch,490
loss,0.05785


Epochs:  0 | Loss:  1.1352792978286743
Epochs:  10 | Loss:  1.1169235706329346
Epochs:  20 | Loss:  1.1063090562820435
Epochs:  30 | Loss:  1.1005470752716064
Epochs:  40 | Loss:  1.084696888923645
Epochs:  50 | Loss:  1.0750772953033447
Epochs:  60 | Loss:  1.0484012365341187
Epochs:  70 | Loss:  1.0274956226348877
Epochs:  80 | Loss:  1.0257288217544556
Epochs:  90 | Loss:  0.9917985796928406
Epochs:  100 | Loss:  0.9348236918449402
Epochs:  110 | Loss:  0.9095746278762817
Epochs:  120 | Loss:  0.862587034702301
Epochs:  130 | Loss:  0.802283525466919
Epochs:  140 | Loss:  0.7580873370170593
Epochs:  150 | Loss:  0.73628830909729
Epochs:  160 | Loss:  0.6659534573554993
Epochs:  170 | Loss:  0.6741551160812378
Epochs:  180 | Loss:  0.6196578741073608
Epochs:  190 | Loss:  0.6003469824790955
Epochs:  200 | Loss:  0.553282618522644
Epochs:  210 | Loss:  0.5794814825057983
Epochs:  220 | Loss:  0.5639512538909912
Epochs:  230 | Loss:  0.5201820135116577
Epochs:  240 | Loss:  0.514018774